<br>

# Introdução


In [ ]:
#!pip3 install traquitanas --upgrade

In [ ]:
import pprint
import time

import pandas as pd
from paths import adds_path, driver_path, logs_path, output_path
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.by import By

from pyFDBS.scraper import FBDS, webdriver

In [ ]:
# from traquitanas.scrapping import adds, gecko
# from scripts.scrapping.driver import Driver
# from selenium.webdriver.support import expected_conditions as EC
# from selenium.webdriver.firefox.options import Options as FirefoxOptions
# from selenium.webdriver.firefox.service import Service as FirefoxService
# input_path_down,; input_path_tab_temp,
# from selenium.webdriver.support.ui import WebDriverWait
# input_path
# from selenium import webdriver

<br>

---

## Driver


In [ ]:
driver = webdriver.Firefox(
    verify_ssl=True,
    download_path=output_path,
)


# driver = webdriver.Chrome(headless=False)


# driver = Driver(
#     my_driver_path=driver_path,
#     my_logs_path=logs_path,
#     my_download_path=input_path_down,
#     verify_ssl=True,
# )

In [ ]:
# # Gecko
# gecko_path = gecko.get_path_geckodriver(driver_path)

# # Logs
# logs_filepath = logs_path / 'geckodriver.log'

# # Services
# service = FirefoxService(executable_path=gecko_path, log_path=logs_filepath)

# # Options
# options = FirefoxOptions()
# options.headless = False
# options.set_preference('intl.accept_languages', 'pt-BR, pt')
# options.set_preference('browser.download.folderList', 2)
# options.set_preference('browser.download.manager.showWhenStarting', False)
# options.set_preference('browser.download.dir', input_path_down.as_posix())
# options.set_preference(
#     'browser.helperApps.neverAsk.saveToDisk',
#     'application/octet-stream, application/pdf, application/vnd.ms-excel',
# )


# # Create Driver
# driver = webdriver.Firefox(service=service, options=options)
# driver.maximize_window()

# # Add Extension
# driver = adds.add_extension_xpath(driver, adds_path)

<br>

--------

# Run


<br>

---

## Abordagem 1


In [ ]:
fbds = FBDS(driver=driver, uf="AC")

Inicialmente listamos os municípios de um estado.


In [ ]:
# Initial Parameters
list_all_files = []

# Parametros
wait_seconds = 3


def loop():
    """
    Lista todos os arquivos recursivamente
    Lista todas as pastas e:
    - se Nº Pastas > 0, entra na pasta e... repete;
    - se Nº Pastas = 0, retorna pra pasta anterior;
    """

    # Lista arquivos
    list_files = fbds.list_itens_h5ai()

    # Se tem Arquivos, junta na lista geral
    if len(list_files) > 0:
        list_all_files.extend(list_files)
        print(f" > Lista geral está com: {len(list_all_files)} arquivos")

    else:
        print(" > Pasta sem arquivos")

    # Lista Pastas
    _list_subfolders = fbds.list_folders_h5ai()

    if len(_list_subfolders) > 0:
        for subfolder in _list_subfolders:
            # Parâmetros
            subfolder_name = subfolder["name"]
            print(f".\\{subfolder_name}")

            # Entra na Pasta
            fbds.find_folder_by_name(subfolder_name)

            # Recursive
            loop()

            # Cria e Salva Tabela
            fbds.save_dataframe(list_all_files, output_path)

            # Sai da Pasta
            fbds.click_back()
            time.sleep(wait_seconds)
    else:
        print(" > Pasta sem subpastas...")

In [ ]:
list_municipios = fbds.get_list_municipios()

Depois fazemos o _download_


In [ ]:
# Lista Pastas
list_folders = fbds.list_folders_h5ai()

for folder in list_folders:
    # Parâmetros
    nome_municipio = folder['name']

    if nome_municipio in list_municipios:
        # Prints
        print(f'Processando letra {nome_municipio}')

        # Entra na Pasta
        fbds.find_folder_by_name(nome_municipio)

        # Recursive
        loop()

        # Sai da Pasta
        fbds.click_back()
        time.sleep(wait_seconds)

# Fim
print('Fim')

<br>

---

## Abordagem 2


In [ ]:
# Driver
#get_fbds()

In [ ]:
# Lista dos Município
list_municipios = fbds.get_list_municipios()
list_municipios = list_municipios[50:52]
list_municipios

In [ ]:
# Action
a = ActionChains(driver)

# Conteúdo Principal
content_xpath = driver.find_element(By.XPATH, "//*[@id='content']")

# Lista
list_folders = content_xpath.find_elements(
    By.XPATH,
    ".//*[@class='item folder' or @class='item folder selected' or @class='item folder folder-parent']",
)

# Results
print(f'São {len(list_folders)} pastas')
pprint.pprint(list_folders[:2])

In [ ]:
# list_folders = list_folders[:3]

In [ ]:
driver.download_path

In [ ]:
list_data = []

for count, folder in enumerate(list_folders):
    #
    infos = fbds.get_dict(folder)
    nome_municipio = infos['name']

    # ddddd
    if nome_municipio in list_municipios:
        # Prints
        print(f'Processando letra {nome_municipio}')

        # URL
        href_xpath = folder.find_element(By.XPATH, ".//a")
        href_value = href_xpath.get_attribute('href')

        # Tipo
        icon_xpath = folder.find_element(
            By.XPATH, ".//a//span[@class='icon square']//img"
        )
        icon_value = icon_xpath.get_attribute('alt')

        # Nome
        label_xpath = folder.find_element(
            By.XPATH, ".//a//span[@class='label']"
        )
        label_value = label_xpath.text

        # Data
        date_xpath = folder.find_element(By.XPATH, ".//a//span[@class='date']")
        date_value = date_xpath.text

        # Size
        size_xpath = folder.find_element(By.XPATH, ".//a//span[@class='size']")
        size_value = size_xpath.text

        # Selector
        selector_xpath = folder.find_element(
            By.XPATH, ".//a//span[@class='selector']"
        )

        # Dictionary
        dict_data = {
            'url': href_value,
            'type': icon_value,
            'name': label_value,
            'date': date_value,
            'size': size_value,
        }
        list_data.append(dict_data)

        # Vai pro Elemento
        print(f'Início de download para {href_value}')
        time.sleep(1)

        # Xpath do Ícone Anterior e move pra lá
        folder_anterior = list_folders[count - 1]
        icon_xpath_anterior = folder_anterior.find_element(
            By.XPATH, ".//a//span[@class='icon square']//img"
        )
        driver.execute_script(
            "return arguments[0].scrollIntoView();",
            icon_xpath_anterior,
        )
        time.sleep(1)

        # Passa o Mouse no Ícone da Pasta Atual
        a.move_to_element(icon_xpath).perform()
        time.sleep(3)

        # Passa o Mouse no Href da Pasta Atual
        a.move_to_element(href_xpath).perform()
        time.sleep(1)

        # Passa o Mouse no Selector da Pasta Atual e Clica!
        a.move_to_element(selector_xpath).click().perform()
        time.sleep(4)

        # Faz Download
        fbds.click_download()
        time.sleep(4)

        # Passa o Mouse no Href da Pasta Atual
        a.move_to_element(icon_xpath).perform()
        time.sleep(2)

        # Passa o Mouse no Selector da Pasta Atual e Clica!
        a.move_to_element(selector_xpath).click().perform()
        time.sleep(2)

# dddd
list_data

<br>

---

## Pastas Atuais


In [ ]:
# Lista todos os arquivos tar e, consequentemente, os municípios com download realizado
list_downloads = list(input_path_down.rglob('*.tar'))
list_downloads = [f.stem for f in list_downloads]
list_downloads = list(set(list_downloads))
list_downloads.sort()
#list_downloads

# Resultados: Nº downloads
print(f'São {len(list_downloads)} downloads')
pprint.pprint(list_downloads[:5])

In [ ]:
# Defini Municípios Faltantes
list_municipios = list(set(list_municipios).difference(list_downloads))
list_municipios.sort()
list_municipios